# Dataset Analysis - Cataract Surgery
This notebook analyzes the dataset from two sources:
- `labels.json` generated by `build_dataset.py`
- Raw folders in `video_matching_json_ayushi`

In [ ]:
import os
import json
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from collections import Counter, defaultdict

## Configuration

In [ ]:
# Paths - adapt to your setup
LABELS_JSON      = "/home/helena/UCL_video_cataract/dataset_final/labels.json"
DATASET_DIR      = "/home/helena/UCL_video_cataract/dataset_final"
SOURCE_DIR       = "/home/helena/UCL_video_cataract/video_matching_json_ayushi"
GRADING_DIR      = "/home/helena/UCL_video_cataract/USB_Ayushi_video_cataract_clean"

## 1. Load labels.json

In [ ]:
with open(LABELS_JSON, "r") as f:
    labels = json.load(f)

# Build a dataframe from labels.json
rows = []
for path, phase in labels.items():
    parts = path.split("/")
    split_name = parts[0]   # train / val / test
    video      = parts[1]
    filename   = parts[2]
    rows.append({"split": split_name, "video": video, "filename": filename, "phase": phase})

df = pd.DataFrame(rows)
print(f"Total frames: {len(df)}")
df.head()

## 2. Load grading mapping from raw folders

In [ ]:
def build_grading_mapping(grading_root):
    mapping = {}
    for folder in os.listdir(grading_root):
        folder_path = os.path.join(grading_root, folder)
        if not os.path.isdir(folder_path):
            continue
        match = re.search(r'(\d+)', folder)
        if not match:
            continue
        level = match.group(1)
        for f in os.listdir(folder_path):
            if f.endswith(".mp4"):
                name = os.path.splitext(f)[0].lower().replace(" ", "").replace("_", "")
                mapping[name] = level
    return mapping

def extract_level(video_name, grading_mapping):
    normalized = video_name.lower().replace(" ", "").replace("_", "")
    for key, level in grading_mapping.items():
        if key in normalized or normalized in key:
            return level
    return "unknown"

grading_mapping = build_grading_mapping(GRADING_DIR)
df["grading"] = df["video"].apply(lambda v: extract_level(v, grading_mapping))

print("Grading distribution:")
print(df["grading"].value_counts())

## 3. Phase distribution per split

In [ ]:
splits = ["train", "val", "test"]
phases = sorted(df["phase"].unique())

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)

for ax, split_name in zip(axes, splits):
    subset = df[df["split"] == split_name]
    counts = subset["phase"].value_counts()
    ax.bar(counts.index, counts.values, color="steelblue")
    ax.set_title(f"{split_name.upper()} - {len(subset)} frames")
    ax.set_xlabel("Phase")
    ax.set_ylabel("Number of frames")
    ax.tick_params(axis='x', rotation=45)

plt.suptitle("Phase distribution per split", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(DATASET_DIR, "phase_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()

## 4. Grading distribution per split

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)

for ax, split_name in zip(axes, splits):
    subset = df[df["split"] == split_name]
    # Count unique videos per grading (not frames)
    video_grading = subset.drop_duplicates(subset="video")[["video", "grading"]]
    counts = video_grading["grading"].value_counts().sort_index()
    ax.bar([f"Grade {g}" for g in counts.index], counts.values, color="coral")
    ax.set_title(f"{split_name.upper()} - {len(video_grading)} videos")
    ax.set_xlabel("Grading")
    ax.set_ylabel("Number of videos")

plt.suptitle("Grading distribution per split (video level)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(DATASET_DIR, "grading_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()

## 5. Analysis from raw source folders

In [ ]:
# Scan source dir to count frames per phase per video
source_rows = []
for video in os.listdir(SOURCE_DIR):
    video_path = os.path.join(SOURCE_DIR, video)
    if not os.path.isdir(video_path):
        continue
    grading = extract_level(video, grading_mapping)
    for phase in os.listdir(video_path):
        phase_path = os.path.join(video_path, phase)
        if not os.path.isdir(phase_path):
            continue
        n_frames = len([f for f in os.listdir(phase_path) if f.endswith(".png")])
        source_rows.append({"video": video, "phase": phase, "grading": grading, "n_frames": n_frames})

df_source = pd.DataFrame(source_rows)
print(f"Total videos in source: {df_source['video'].nunique()}")
print(f"Total phases found: {df_source['phase'].nunique()}")
print(f"Total frames: {df_source['n_frames'].sum()}")
df_source.head()

In [ ]:
# Frames per phase globally (source)
phase_totals = df_source.groupby("phase")["n_frames"].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(phase_totals.index, phase_totals.values, color="mediumseagreen")
ax.set_title("Total frames per phase (all videos)", fontsize=13, fontweight="bold")
ax.set_xlabel("Phase")
ax.set_ylabel("Number of frames")
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(DATASET_DIR, "frames_per_phase_global.png"), dpi=150, bbox_inches="tight")
plt.show()

## 6. Visualize example frames per phase

In [ ]:
N_EXAMPLES = 3  # number of example frames per phase

for phase in sorted(df["phase"].unique()):
    phase_df = df[df["phase"] == phase]
    samples = phase_df.sample(min(N_EXAMPLES, len(phase_df)), random_state=42)

    fig, axes = plt.subplots(1, len(samples), figsize=(5 * len(samples), 4))
    if len(samples) == 1:
        axes = [axes]

    fig.suptitle(f"Phase: {phase}", fontsize=13, fontweight="bold")

    for ax, (_, row) in zip(axes, samples.iterrows()):
        img_path = os.path.join(DATASET_DIR, row["split"], row["video"], row["filename"])
        if os.path.exists(img_path):
            img = mpimg.imread(img_path)
            ax.imshow(img)
            ax.set_title(f"{row['split']}\n{row['video'][:30]}", fontsize=8)
        else:
            ax.set_title("File not found")
        ax.axis("off")

    plt.tight_layout()
    plt.show()